# 01a｜首批真实语料与 8K 词表

130U-FinLLM｜130U 金融大语言模型

下载固定版本的限量 Wikipedia 和 SEntFiN 候选标题，保存带日期和哈希的原创 FOMC 纪要快照。先冻结全部候选任务原文，再自训词表和分片。此页不会把新闻情绪自动改成看多/看空标签，也不会启动模型训练。预计原始文本为几十 MB，下载库仍有额外缓存。

每个 Notebook 都包含独立环境入口。本页准备语料可使用 CPU；下一本 02a 训练前再切换 T4 GPU。点击执行代码前可查看源码；所有输出均为实际执行后产生，仓库不预填训练成绩。

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
command(sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<3", *tokenizer_requirements, "scikit-learn>=1.5,<2")
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps")
check_tokenizer_compatibility(tokenizer_requirements)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive 项目目录：", DRIVE_ROOT)

本页默认使用 seed_v1。重复运行时先校验并复用完整成果；新运行时会恢复 Drive 中本版本的快照。缺失或被修改的成果会明确报错并保留，届时选择新版本号或恢复完整备份。原始材料仅保存在当前运行时及你自己的 Drive，不上传 GitHub。

In [ ]:
SEED_VERSION = "seed_v1"
ROOT = Path("data/raw") / SEED_VERSION
CORPORA = ROOT / "corpora"
TASK = ROOT / "task"
PREPARED = ROOT / "prepared"
TOKENIZER = Path("tokenizer") / SEED_VERSION
PROCESSED = Path("data/processed") / SEED_VERSION
RUN_OUTPUT = Path("runs") / (SEED_VERSION + "_pretrain")
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]", cwd=PROJECT)
SEED_DRIVE = DRIVE_ROOT / "seeds" / SEED_VERSION
for name, destination in (("raw", ROOT), ("processed", PROCESSED), ("tokenizer", TOKENIZER)):
    source = SEED_DRIVE / name
    if not destination.exists() and source.exists():
        require(source / "SNAPSHOT_COMPLETE.json")
        finllm("sync", "--source", source, "--destination", PROJECT / destination, "--working-copy")
SEED_BUILD_ARGS = ["--corpora", CORPORA, "--task-pool", TASK / "task_exclusions.jsonl",
                   "--prepared", PREPARED, "--tokenizer", TOKENIZER,
                   "--processed", PROCESSED, "--run-output", RUN_OUTPUT]
print("本页只准备数据；当前 CUDA 可用：", torch.cuda.is_available())

In [ ]:
command(sys.executable, "-u", "scripts/fetch_seed_corpora.py", "--output", CORPORA,
        "--general-max-bytes", "20000000", cwd=PROJECT)
command(sys.executable, "-u", "scripts/prepare_sentfin_review.py", "--download", "--output", TASK, cwd=PROJECT)

In [ ]:
command(sys.executable, "-u", "scripts/build_seed_artifacts.py", *SEED_BUILD_ARGS, cwd=PROJECT)
print((PROCESSED / "seed_summary.json").read_text(encoding="utf-8"))

In [ ]:
# 备份前重新检查完整产物，不能把中断执行的半成品显示为成功。
command(sys.executable, "-u", "scripts/build_seed_artifacts.py", *SEED_BUILD_ARGS, "--validate-only", cwd=PROJECT)

SEED_DRIVE = DRIVE_ROOT / "seeds" / SEED_VERSION
for name, source in (("raw", Path("data/raw") / SEED_VERSION),
                     ("processed", Path("data/processed") / SEED_VERSION),
                     ("tokenizer", Path("tokenizer") / SEED_VERSION),
                     ("run", Path("runs") / (SEED_VERSION + "_pretrain"))):
    if source.exists():
        finllm("sync", "--source", PROJECT / source, "--destination", SEED_DRIVE / name)
shutil.copy2(PROJECT / "runtime_revision.txt", SEED_DRIVE / "runtime_revision.txt")
print("本版本备份完成：", SEED_DRIVE)

完成后先查看 seed_summary.json 的真实 token 数。200 条任务审核表在 data/raw/seed_v1/task，填写的是你对目标资产立场的判断；原数据作者的人类情绪标签保留在单独列中。下一本 02a 可使用本页产物运行 24M 模型的约 1M-token 预训练试跑。